# 017 — Juegos: minimax y poda alfa-beta

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("search", seed=17)
assert result["kind"] == "search"
assert result["evidence"]
show(result)


## Solución 1 — Minimax

`B = min(3, 12, 8) = 3`; `C = min(2, 4, 6) = 2`; `D = min(14, 5, 2) = 2`;
raíz = `max(3, 2, 2) = 3` → **mover a B**. La elección asume que MIN jugará
óptimamente: si MIN se equivocara, MAX solo puede obtener ≥ 3 — el valor
minimax es una **garantía**, no una predicción.


## Solución 2 — Alfa-beta

```text
B: hojas 3, 12, 8 → B = 3; raíz: α = 3
C: primera hoja 2 → β_C = 2 ≤ α = 3  →  PODA hojas 4 y 6
D: hoja 14 (β=14), hoja 5 (β=5), hoja 2 (β=2 ≤ α... llega tras la última hoja)
   D = 2, sin poda efectiva en D
raíz = max(3, ≤2, 2) = 3
```

a) Se podan `4` y `6`: en cuanto C muestra un 2, MIN puede forzar ≤ 2 ahí,
y MAX ya tiene 3 garantizado en B — C es irrelevante.
b) Se evalúan **7 de 9** hojas.


## Solución 3 — Orden y poda

a) Orden D, C, B: D = 2 → α = 2; C: hoja 2 → β = 2 ≤ α → poda 4, 6 (2 hojas
evaluadas... la primera); B: hojas 3, 12, 8 sin poda (β nunca cae bajo α antes
del final: 3 > 2). Total ≈ 8 hojas: ¡peor que el orden original!

b) El mejor orden empieza por **B (el mejor hijo, valor 3)**: B completo
(3 hojas, α=3), C poda tras su primera hoja (2 ≤ 3), y D poda tras su tercera...
con D=[14,5,2] las dos primeras no bajan de α: se evalúan 14, 5, 2. Si dentro
de D el 2 fuera primero, D podaría de inmediato: orden B, C, D con hojas de D
reordenadas [2,…] evalúa solo 3+1+1 = 5 hojas.

c) Regla práctica: **examinar primero el movimiento probablemente mejor**
(killer moves, tablas de transposición): la poda máxima `O(b^(d/2))` solo se
alcanza con buena ordenación.


In [ ]:
def alfabeta(nodo, es_max, alfa, beta, contador):
    if isinstance(nodo, (int, float)):
        contador[0] += 1
        return nodo
    if es_max:
        v = float("-inf")
        for hijo in nodo:
            v = max(v, alfabeta(hijo, False, alfa, beta, contador))
            alfa = max(alfa, v)
            if alfa >= beta:
                break
        return v
    v = float("inf")
    for hijo in nodo:
        v = min(v, alfabeta(hijo, True, alfa, beta, contador))
        beta = min(beta, v)
        if alfa >= beta:
            break
    return v

for orden, arbol in {
    "B,C,D": [[3, 12, 8], [2, 4, 6], [14, 5, 2]],
    "D,C,B": [[14, 5, 2], [2, 4, 6], [3, 12, 8]],
    "B,C,D con D=[2,14,5]": [[3, 12, 8], [2, 4, 6], [2, 14, 5]],
}.items():
    contador = [0]
    v = alfabeta(arbol, True, float("-inf"), float("inf"), contador)
    print(f"orden {orden}: valor {v}, hojas evaluadas {contador[0]}")


## Solución 4 — De búsqueda a juego

Habría que añadir: (1) un **segundo agente** que mueve en turnos alternos, con
lo que `RESULT` depende de quién juega; (2) una **utilidad** en los estados
terminales en lugar de un test de objetivo (ganar/perder/empatar o un
marcador); (3) el criterio de decisión minimax en lugar del costo mínimo. La
solución deja de ser un camino porque MIN elige la mitad de las transiciones:
lo que se calcula es una **estrategia** (qué responder a cada réplica), no una
secuencia fija de acciones.


## Reflexión

1. En el árbol de referencia, alfa-beta poda las hojas 4 y 6 pero ninguna de D. ¿Qué reordenación de los hijos de la raíz maximizaría la poda y por qué el 'mejor movimiento primero' es la regla general?
2. Alfa-beta devuelve exactamente el mismo valor que minimax. ¿De dónde sale entonces la ganancia práctica de b^d a b^(d/2), y de qué depende alcanzarla?
3. Si las hojas fueran valores de una función de evaluación truncada (no utilidades exactas), ¿qué significa el 'valor 3' de la raíz y qué riesgo introduce el efecto horizonte?
